# Phase 2 — Seed data + competency-question queries

Loads the ontology **and** the seed fixture into one graph, runs OWL-RL inference, then answers three competency questions. Watch the reasoner derive `dependsOn` (transitive), `runsOn`, and `mayImpactFunction` — none of which are asserted.

In [ ]:
from pathlib import Path
from rdflib import Graph
import owlrl

SSC  = 'http://systemscyber.colostate.edu/ssc#'
DATA = 'http://systemscyber.colostate.edu/ssc/data#'
PREFIXES = f'PREFIX ssc: <{SSC}>\nPREFIX : <{DATA}>\n'

g = Graph()
g.parse('../ontology-ssc/ssc.ttl', format='turtle')  # schema + axioms
g.parse('../data/seed.ttl', format='turtle')            # instances
before = len(g)
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)
print(f'Triples: {before} -> {len(g)} after inference')

In [2]:
def run(title, q):
    print(f'== {title} ==')
    for row in g.query(PREFIXES + q):
        print('  ', ' | '.join(str(x).split('#')[-1] for x in row))
    print()

In [3]:
# CQ2 — transitive dependencies of app_ivi (openssl asserted; zlib should be INFERRED)
run('CQ2  transitive dependencies of :app_ivi',
    'SELECT ?dep WHERE { :app_ivi ssc:dependsOn ?dep } ORDER BY ?dep')

== CQ2  transitive dependencies of :app_ivi ==
   openssl_3_0_8
   zlib_1_2_11



In [4]:
# CQ7 — which components are affected by CVE-2022-37434, and with what confidence
run('CQ7  components affected by CVE-2022-37434',
    '''SELECT ?comp ?conf WHERE {
         ?m a ssc:ComponentVulnerabilityMatch ;
            ssc:matchesVulnerability :CVE-2022-37434 ;
            ssc:matchesComponent ?comp ;
            ssc:hasConfidence ?conf . }''')

== CQ7  components affected by CVE-2022-37434 ==
   zlib_1_2_11 | 1.0



In [5]:
# Inferred runsOn (component -> device) via includedIn o deployedOn
run('runsOn  (inferred: component runs on device)',
    'SELECT ?comp ?dev WHERE { ?comp ssc:runsOn ?dev } ORDER BY ?comp')

== runsOn  (inferred: component runs on device) ==
   app_ivi | uthp_gateway
   openssl_3_0_8 | uthp_gateway
   zlib_1_2_11 | uthp_gateway



In [6]:
# CQ10 — which components MAY IMPACT braking (the network pivot, fully inferred)
run('CQ10  components that may impact :braking',
    'SELECT ?comp WHERE { ?comp ssc:mayImpactFunction :braking } ORDER BY ?comp')

== CQ10  components that may impact :braking ==
   app_ivi
   openssl_3_0_8
   zlib_1_2_11



## What just happened
- **CQ2** returned `zlib_1_2_11` even though only `app_ivi -> openssl -> zlib` was asserted (transitive `dependsOn`).
- **runsOn** and **CQ10** are fully derived by the property chains — the vulnerable `zlib` on the UTHP gateway is inferred to *may impact braking* through `deployedOn -> connectsTo -> reaches -> supports`.
- This is the reasoning an ungrounded LLM or a flat CVE scanner cannot reproduce — the basis of the ablation.